# 02. 에이전트 평가 (Agent Evaluation)

> 단위 테스트가 못 잡는 품질 회귀는 **평가 데이터셋 + LLM-as-Judge** 가 잡아요. LangSmith 데이터셋과 openevals · agentevals 로 자동 평가 파이프라인을 구축해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. 에이전트 평가의 3가지 전략(최종 응답, 궤적, 단일 스텝)을 구별하고 적절히 선택할 수 있어요
2. `agentevals`로 궤적 매칭 평가자(strict/unordered/subset/superset)를 구성하고 실행할 수 있어요
3. `openevals`로 LLM-as-Judge 평가자를 만들어 에이전트 응답의 정확도를 측정할 수 있어요
4. LangSmith `client.evaluate()`로 데이터셋 기반 전체 평가 파이프라인을 구성하고 실행할 수 있어요

## 사전 지식

- `01-Agent-Testing.ipynb` — FakeChat 기반 단위 테스트, 통합 테스트
- LangSmith 기초 (프로젝트, 트레이싱 설정)
- LangGraph 에이전트 기본 구조 (StateGraph, 도구 호출)

## 왜 에이전트 평가가 필요한가?

단위 테스트와 통합 테스트가 「코드가 올바르게 실행되는가?」를 검증한다면, 에이전트 평가는 「이 에이전트가 **좋은 결정**을 내리는가?」를 측정해요.

이것은 마치 **요리사 자격시험**과 같아요. 요리사가 칼을 쓸 줄 안다(단위 테스트), 주방 장비가 작동한다(통합 테스트)는 것만으로는 부족해요. 실제로 **맛있는 요리를 만들어내는지**(평가), **올바른 순서로 조리하는지**(궤적 평가)를 검증해야 해요.

> 🎯 **강의 포인트**: 테스트와 평가의 차이를 명확히 구분하세요. 테스트는 **버그를 찾는 것**, 평가는 **품질을 측정하는 것**이에요. 테스트는 pass/fail이지만, 평가는 0.0~1.0 사이의 점수예요.

## LangSmith 평가 핵심 용어

공식 문서([Evaluation Concepts](https://docs.langchain.com/langsmith/evaluation-concepts.md))는 평가 파이프라인을 네 가지 1급 개념으로 정의해요. 이후 셀에서 사용할 API들의 어휘가 전부 여기서 나와요.

| 용어 | 공식 정의 | 이 노트북에서 등장하는 곳 |
|------|-----------|-------------------------|
| **Dataset** | 평가용 Example의 모음(collection) | `client.create_dataset(...)` |
| **Example** | 입력(`inputs`) + 참조 출력(`outputs`) + 메타데이터로 구성된 개별 테스트 케이스 | `client.create_examples(...)` |
| **Experiment** | 특정 애플리케이션 버전을 Dataset으로 평가한 결과 1회분. 점수·출력·트레이스를 모두 담아요 | `client.evaluate(...)` 반환값 |
| **Evaluator** | 애플리케이션 출력을 채점하는 함수·평가자 리소스 | `evaluate_correctness`, `evaluate_trajectory_*` 등 |
| **LLM-as-a-judge** | LLM을 심판으로 사용해 출력을 채점하는 Evaluator의 한 종류 | `openevals.create_llm_as_judge(...)` |
| **Trajectory** | 에이전트가 거친 도구 호출과 인자 포맷의 시퀀스 | `agentevals.create_trajectory_match_evaluator(...)` |

> 🔑 **핵심 흐름**: Dataset에서 Example을 하나씩 꺼내 애플리케이션(에이전트)에 입력하고, 그 출력을 Evaluator들이 채점한 결과가 Experiment로 기록돼요. LangSmith Studio에서는 Experiment끼리 점수를 비교해 어느 버전이 더 나은지 확인할 수 있어요.

## 에이전트 평가란?

에이전트 평가(Agent Evaluation)는 에이전트가 올바르게 작동하는지를 **데이터셋 기반으로 자동 측정**하는 과정이에요.

```mermaid
flowchart TD
    DS[(데이터셋<br/>Dataset)] --> |입력 + 정답| E[평가 실행<br/>client.evaluate]
    E --> |입력 전달| AG[에이전트<br/>Agent]
    AG --> |출력 반환| EV[평가자<br/>Evaluators]
    DS --> |정답 참조| EV
    EV --> |점수 기록| LS[LangSmith<br/>실험 결과]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class DS input
    class AG,E process
    class EV output
    class LS storage
```

### 3가지 평가 전략

| 전략 | 비유 | 평가 대상 | 사용 시점 | 대표 도구 |
|------|------|-----------|-----------|----------|
| **최종 응답 평가** | 요리 맛 평가 | 에이전트의 최종 텍스트 응답 | 응답 품질 측정 (정확도, 유용성) | `openevals` LLM-as-Judge |
| **궤적 평가** | 조리 과정 평가 | 에이전트가 거친 도구 호출 순서 전체 | 에이전트가 올바른 경로로 문제를 풀었는지 | `agentevals` 궤적 매처 |
| **단일 스텝 평가** | 개별 조리 단계 평가 | 특정 노드 1개의 결정 | 특정 라우팅/도구 선택 검증 | 커스텀 평가자 |

> 🔑 **핵심 개념**: 에이전트는 단순 함수가 아니에요. 여러 단계를 거쳐 최종 응답을 만들기 때문에, 최종 결과뿐 아니라 **어떤 경로로 도달했는지(궤적)**도 평가해야 해요. 정답을 맞혔더라도 불필요한 도구를 10번 호출했다면 비효율적인 에이전트예요.

### 주요 패키지

| 패키지 | 용도 | 비유 |
|--------|------|------|
| `langsmith` | 데이터셋 관리, 평가 실행, 결과 기록 | 시험장 + 성적표 |
| `agentevals` | 에이전트 궤적(도구 호출 시퀀스) 평가 | 조리 과정 채점표 |
| `openevals` | LLM-as-Judge 평가 (정확도, 품질) | AI 미식 심사위원 |

## 환경 설정

In [1]:
# 환경 변수 로드 (.env 파일에서 API 키를 읽어요)
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os

# LangSmith 추적 설정: 평가 결과를 LangSmith에 기록해요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Evaluation"

# LANGSMITH_API_KEY는 .env 파일에 설정되어 있어야 해요
print("LangSmith 추적 활성화:", os.environ.get("LANGCHAIN_TRACING_V2"))
print("프로젝트:", os.environ.get("LANGCHAIN_PROJECT"))

LangSmith 추적 활성화: true
프로젝트: None


## 1. 평가 대상 에이전트 준비

평가를 시작하기 전에 평가할 에이전트를 만들어요.
여기서는 간단한 검색 도구를 사용하는 에이전트를 사용해요.

> 💡 **실무 팁**: 실제 프로젝트에서는 이미 만들어진 에이전트를 import해서 사용하면 돼요. 평가 코드는 에이전트 코드와 분리해서 관리하는 게 좋아요.

In [3]:
# 에이전트 구성에 필요한 패키지를 가져와요
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 기본 모델 설정 (비용 효율적인 gpt-4o-mini 사용)
# temperature=0: 평가는 "재현 가능"해야 하므로 에이전트의 무작위성을 제거해요.
# (temperature가 높으면 같은 질문에도 도구 선택·응답이 흔들려 점수가 매번 달라져요)
model = init_chat_model("openai:gpt-4o-mini", temperature=0)

In [4]:
# ---------------------------------------------------
# 평가용 도구 정의
# ---------------------------------------------------
# 실제 API 대신 결정적(deterministic) 응답을 반환하는
# 테스트용 도구를 정의해요. 평가 재현성을 위해 중요해요.
# 도구가 3개로 늘면 "어떤 도구를 골라 호출하는가"를 평가하는 궤적 평가가 더 흥미로워져요.

@tool
def search_weather(city: str) -> str:
    """주어진 도시의 현재 날씨를 조회해요."""
    # 테스트용 고정 데이터 (실제 API 대신)
    weather_data = {
        "서울": "맑음, 기온 22도",
        "부산": "흐림, 기온 25도",
        "제주": "비, 기온 20도",
        "seoul": "맑음, 기온 22도",
        "busan": "흐림, 기온 25도",
    }
    # 대소문자 구분 없이 검색 (데이터에 없는 도시는 "찾을 수 없어요"를 반환)
    city_lower = city.lower()
    return weather_data.get(city_lower, f"{city}의 날씨 정보를 찾을 수 없어요.")


@tool
def get_population(city: str) -> str:
    """주어진 도시의 인구를 조회해요."""
    # 테스트용 고정 데이터
    population_data = {
        "서울": "약 950만 명",
        "부산": "약 330만 명",
        "제주": "약 70만 명",
        "seoul": "약 950만 명",
        "busan": "약 330만 명",
    }
    city_lower = city.lower()
    return population_data.get(city_lower, f"{city}의 인구 정보를 찾을 수 없어요.")


@tool
def get_landmark(city: str) -> str:
    """주어진 도시의 대표 랜드마크(명소)를 조회해요."""
    # 테스트용 고정 데이터
    landmark_data = {
        "서울": "경복궁, 남산서울타워",
        "부산": "해운대, 광안대교",
        "제주": "성산일출봉, 한라산",
        "seoul": "경복궁, 남산서울타워",
        "busan": "해운대, 광안대교",
    }
    city_lower = city.lower()
    return landmark_data.get(city_lower, f"{city}의 랜드마크 정보를 찾을 수 없어요.")


# 에이전트에서 사용할 도구 목록 (날씨·인구·랜드마크 3종)
tools = [search_weather, get_population, get_landmark]
print(f"등록된 도구: {[t.name for t in tools]}")

등록된 도구: ['search_weather', 'get_population', 'get_landmark']


In [5]:
# ---------------------------------------------------
# 에이전트 생성
# ---------------------------------------------------
# create_agent: V1 방식으로 도구 호출 에이전트를 만들어요

agent = create_agent(
    model,
    tools,
    system_prompt="당신은 도시 정보를 제공하는 유용한 어시스턴트예요. 날씨·인구·랜드마크 정보를 정확하게 알려주세요.",
)

# 에이전트 동작 간단 확인
test_result = agent.invoke({"messages": [HumanMessage(content="서울의 날씨를 알려줘")]})
# 에이전트 테스트 응답:
print(test_result["messages"][-1].content)

서울의 현재 날씨는 맑고, 기온은 22도입니다.


## 2. LangSmith 데이터셋 준비

평가를 실행하려면 **입력(input)**과 **기대 출력(reference output)**으로 구성된 데이터셋이 필요해요.

데이터셋은 마치 **시험 문제지**와 같아요. 문제(inputs)와 모범 답안(outputs)을 미리 만들어두면, 에이전트가 시험을 치르고 채점을 받을 수 있어요.

```mermaid
flowchart LR
    subgraph 데이터셋 구조
        E1[예제 1<br/>입력: 서울 날씨?<br/>정답: 맑음 22도 포함] 
        E2[예제 2<br/>입력: 부산 인구?<br/>정답: 330만 포함]
        E3[예제 3<br/>입력: 제주 날씨와 인구?<br/>정답: 두 도구 모두 사용]
    end
    E1 --> DS[(LangSmith<br/>데이터셋)]
    E2 --> DS
    E3 --> DS

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class E1,E2,E3 input
    class DS storage
```

> 🔑 **핵심 개념**: LangSmith 데이터셋의 각 예제(Example)는 `inputs`와 `outputs` 두 부분으로 구성돼요. `inputs`는 에이전트에게 전달되고, `outputs`는 평가자가 비교할 정답이에요.

> 💡 **실무 팁**: 좋은 데이터셋은 **다양한 시나리오**를 포함해야 해요. 단일 도구 호출, 다중 도구 호출, 도구가 필요 없는 경우, 에지 케이스(존재하지 않는 도시 등)를 모두 포함하면 더 신뢰할 수 있는 평가 결과를 얻을 수 있어요.

In [6]:
# LangSmith 클라이언트를 가져와요
from langsmith import Client

# LangSmith 클라이언트 초기화 (API 키는 .env에서 로드)
client = Client()

In [7]:
# ---------------------------------------------------
# 데이터셋 생성
# ---------------------------------------------------
# 같은 이름의 데이터셋이 이미 있으면 삭제 후 재생성해요
# (노트북 재실행 시 중복 생성 방지)

DATASET_NAME = "도시정보-에이전트-평가-v1"

# 기존 데이터셋 확인 및 삭제
existing_datasets = list(client.list_datasets(dataset_name=DATASET_NAME))
if existing_datasets:
    try:
        client.delete_dataset(dataset_id=existing_datasets[0].id)
        print(f"기존 데이터셋 삭제: {DATASET_NAME}")
    except Exception:
        # 이미 삭제되었거나 찾을 수 없는 경우 무시해요
        pass

# 새 데이터셋 생성
dataset = client.create_dataset(
    dataset_name=DATASET_NAME,
    description="도시 날씨·인구 정보 조회 에이전트 평가 데이터셋",
)
print(f"데이터셋 생성 완료: {dataset.name} (ID: {dataset.id})")

데이터셋 생성 완료: 도시정보-에이전트-평가-v1 (ID: 845b0547-f6ff-449c-870f-9c2a003c8148)


In [8]:
# ---------------------------------------------------
# 평가 예제 추가 (점수가 "다양하게" 나오도록 의도적으로 설계)
# ---------------------------------------------------
# inputs: 에이전트에 전달할 입력
# outputs: 평가자가 비교할 기대 출력 (answer = 기대 응답, trajectory = 기대 도구 호출)
#
# 핵심 설계: 모든 예제가 1.0이면 평가의 의미가 없어요. 그래서 일부러
#   - 정답도 궤적도 맞는 "정상" 예제 (4개)
#   - 궤적은 맞지만 정답이 틀리는 예제 (대전: 데이터에 없어 답을 못 찾음)
#   - 정답은 맞지만 궤적이 어긋나는 예제 (서울 전체: 기대보다 많은 도구 호출)
# 을 섞었어요. 두 지표(correctness vs trajectory)가 서로 다른 문제를 잡아내는 걸 보여줘요.

examples = [
    # 1) 정상: 단일 도구 (날씨) — correctness O, trajectory O
    {
        "inputs": {"messages": [{"role": "user", "content": "서울의 날씨를 알려줘"}]},
        "outputs": {
            "answer": "서울은 맑음이고 기온은 22도예요.",
            "trajectory": [
                {"role": "tool", "tool": "search_weather", "tool_input": {"city": "서울"}}
            ],
        },
    },
    # 2) 정상: 단일 도구 (인구) — correctness O, trajectory O
    {
        "inputs": {"messages": [{"role": "user", "content": "부산의 인구가 얼마나 돼?"}]},
        "outputs": {
            "answer": "부산의 인구는 약 330만 명이에요.",
            "trajectory": [
                {"role": "tool", "tool": "get_population", "tool_input": {"city": "부산"}}
            ],
        },
    },
    # 3) 정상: 다중 도구 (날씨 + 인구) — correctness O, trajectory O
    {
        "inputs": {"messages": [{"role": "user", "content": "제주의 날씨와 인구를 모두 알려줘"}]},
        "outputs": {
            "answer": "제주는 비가 오고 기온은 20도이며, 인구는 약 70만 명이에요.",
            "trajectory": [
                {"role": "tool", "tool": "search_weather", "tool_input": {"city": "제주"}},
                {"role": "tool", "tool": "get_population", "tool_input": {"city": "제주"}},
            ],
        },
    },
    # 4) 정상: 새 도구 (랜드마크) — correctness O, trajectory O
    {
        "inputs": {"messages": [{"role": "user", "content": "서울의 대표 랜드마크는 뭐가 있어?"}]},
        "outputs": {
            "answer": "서울은 경복궁과 남산서울타워가 유명해요.",
            "trajectory": [
                {"role": "tool", "tool": "get_landmark", "tool_input": {"city": "서울"}}
            ],
        },
    },
    # 5) [궤적 O, 정답 X] 대전은 도구 데이터에 없어요.
    #    에이전트는 "올바른 도구(search_weather)"를 호출하지만 데이터가 없어 답을 못 줘요.
    #    → 궤적 평가는 통과, 최종 응답(correctness)은 실패. "길은 맞는데 목적지에 못 감".
    {
        "inputs": {"messages": [{"role": "user", "content": "대전의 날씨를 알려줘"}]},
        "outputs": {
            "answer": "대전은 맑고 기온은 18도예요.",  # 기대 정답 (도구엔 없는 정보)
            "trajectory": [
                {"role": "tool", "tool": "search_weather", "tool_input": {"city": "대전"}}
            ],
        },
    },
    # 6) [정답 O, 궤적 X] 기대 궤적은 "핵심 2개 도구"로만 좁게 정의했어요.
    #    그런데 질문이 랜드마크까지 요구해서 에이전트가 도구 3개를 호출해요.
    #    → 최종 응답은 맞지만(correctness O), strict/unordered는 "집합이 다르다"며 실패.
    #      대신 superset(필수 도구를 모두 포함하면 OK)은 통과해요.
    {
        "inputs": {"messages": [{"role": "user", "content": "서울의 날씨, 인구, 랜드마크를 모두 알려줘"}]},
        "outputs": {
            "answer": "서울은 맑고 22도이며, 인구는 약 950만 명이고, 경복궁·남산서울타워가 유명해요.",
            # 기대 궤적을 일부러 2개로만 정의 (superset 모드의 효용을 보기 위함)
            "trajectory": [
                {"role": "tool", "tool": "search_weather", "tool_input": {"city": "서울"}},
                {"role": "tool", "tool": "get_population", "tool_input": {"city": "서울"}},
            ],
        },
    },
]

# 예제를 데이터셋에 추가해요
client.create_examples(
    dataset_id=dataset.id,
    inputs=[ex["inputs"] for ex in examples],
    outputs=[ex["outputs"] for ex in examples],
)

print(f"예제 {len(examples)}개 추가 완료")
print("설계 의도: 정상 4 + 궤적O·정답X(대전) 1 + 정답O·궤적X(서울 전체) 1")

예제 6개 추가 완료
설계 의도: 정상 4 + 궤적O·정답X(대전) 1 + 정답O·궤적X(서울 전체) 1


## 3. 최종 응답 평가: LLM-as-Judge

**최종 응답 평가**는 에이전트가 마지막으로 사용자에게 돌려준 텍스트가 정답과 얼마나 일치하는지 측정해요.

`openevals` 패키지의 `create_llm_as_judge()`는 LLM을 심판으로 사용해서 텍스트 품질을 평가해요.

> 🎯 **강의 포인트**: LLM-as-Judge는 정규 표현식이나 문자열 매칭으로는 판별하기 어려운 의미론적 정확도를 평가할 때 강력해요. "22도"와 "섭씨 22도"는 표현은 다르지만 의미는 같으니까요.

> ⚠️ **자주 하는 실수**: `CORRECTNESS_PROMPT`는 `outputs` (에이전트 응답)과 `reference_outputs` (데이터셋 정답)을 비교해요. 평가자 함수의 파라미터 이름을 정확히 `outputs`, `reference_outputs`로 써야 해요.

In [9]:
# !uv add openevals

In [10]:
# openevals에서 LLM 심판 생성 도구를 가져와요
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

In [11]:
# ---------------------------------------------------
# LLM-as-Judge 평가자 생성
# ---------------------------------------------------
# CORRECTNESS_PROMPT: 응답이 기준 정답에 비해 얼마나 정확한지 평가해요
#
# [재현성 핵심 2가지]
# 1) temperature=0: 심판도 무작위성을 제거해야 같은 입력에 같은 점수가 나와요.
# 2) 더 강한 심판 모델: 평가 대상(gpt-4o-mini)보다 한 단계 높은 gpt-4o를 심판으로 써요.
#    약한 심판(gpt-4o-mini)은 한국어 "만(萬)" 단위를 오독하거나(330만→330,000)
#    자기가 아는 외부 지식을 끌어와 멀쩡한 정답을 틀렸다고 깎는 일이 잦아요.
# → model="..." 문자열 대신, 미리 설정한 모델을 judge= 인자로 넘겨요.
judge_model = init_chat_model("openai:gpt-4o", temperature=0)

correctness_evaluator = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    judge=judge_model,            # 사전 설정한 심판 모델 (gpt-4o, temperature=0)
    feedback_key="correctness",   # LangSmith에 기록될 점수 키 이름
)

# LLM-as-Judge 평가자 생성 완료
print("심판 모델: gpt-4o (temperature=0)")
print("평가 대상: gpt-4o-mini  → 심판이 대상보다 강해야 채점이 안정적이에요")

심판 모델: gpt-4o (temperature=0)
평가 대상: gpt-4o-mini  → 심판이 대상보다 강해야 채점이 안정적이에요


In [12]:
# ---------------------------------------------------
# 평가자 함수 래퍼
# ---------------------------------------------------
# LangSmith evaluate()가 기대하는 형식으로 평가자를 감싸요
# 시그니처: (inputs, outputs, reference_outputs) → {key: score}

def evaluate_correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """에이전트 최종 응답의 정확도를 LLM으로 평가해요.
    
    Args:
        inputs: 데이터셋 예제의 입력 (질문)
        outputs: 에이전트가 실제로 생성한 출력
        reference_outputs: 데이터셋의 기대 출력 (정답)
    
    Returns:
        점수 딕셔너리 {'correctness': 0 또는 1}
    """
    # 에이전트 응답에서 마지막 AI 메시지 추출
    agent_response = outputs.get("messages", [])[-1]
    if hasattr(agent_response, "content"):
        agent_text = agent_response.content  # Message 객체인 경우
    else:
        agent_text = str(agent_response)     # 딕셔너리인 경우
    
    # 입력 질문 추출
    input_messages = inputs.get("messages", [])
    input_text = ""
    if input_messages:
        first_msg = input_messages[0]
        if isinstance(first_msg, dict):
            input_text = first_msg.get("content", "")
        elif hasattr(first_msg, "content"):
            input_text = first_msg.content
    
    # LLM 심판으로 정확도 평가 (inputs, outputs, reference_outputs 모두 전달)
    result = correctness_evaluator(
        inputs={"question": input_text},
        outputs={"response": agent_text},
        reference_outputs={"answer": reference_outputs.get("answer", "")},
    )
    return result


# 단독 실행 테스트
test_input = {"messages": [{"role": "user", "content": "서울 날씨 알려줘"}]}
test_output = {"messages": [{"content": "서울은 현재 맑으며 기온은 22도입니다."}]}
test_reference = {"answer": "서울은 맑음이고 기온은 22도예요."}

test_score = evaluate_correctness(test_input, test_output, test_reference)
print(f"테스트 정확도 점수: {test_score}")

테스트 정확도 점수: {'key': 'correctness', 'score': True, 'comment': "The model's output provides the current weather condition in Seoul as 'clear' and the temperature as 22 degrees Celsius. This matches the reference output, which also states that Seoul is 'clear' and the temperature is 22 degrees Celsius. The response is accurate, complete, and uses precise terminology. There are no factual errors, misleading statements, or logical inconsistencies. Thus, the score should be: true.", 'metadata': None}


## 4. 궤적 평가: agentevals

**궤적 평가(Trajectory Evaluation)**는 에이전트가 어떤 도구를 어떤 순서로 호출했는지를 평가해요.

최종 응답이 맞더라도 엉뚱한 도구를 사용했다면 문제가 있을 수 있으니까요. 이것은 마치 **내비게이션 경로 평가**와 같아요. 목적지(정답)에 도착했더라도 고속도로 대신 비포장도로를 달렸다면 좋은 경로가 아닌 거예요.

```mermaid
flowchart TD
    subgraph 기대 궤적
        E1[search_weather 호출] --> E2[get_population 호출]
    end
    subgraph 실제 궤적
        A1[search_weather 호출] --> A2[get_population 호출]
    end
    A1 -->|비교| EV{궤적 매처}
    E1 -->|비교| EV
    EV -->|match| SC[점수 1.0]
    EV -->|mismatch| SF[점수 0.0]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class E1,E2 input
    class A1,A2 process
    class EV output
    class SC output
    class SF error
```

### 궤적 매칭 모드 4가지

| 모드 | 비유 | 의미 | 적합한 상황 |
|------|------|------|-------------|
| `strict` | 정확한 레시피 순서 | 도구 이름 + 인자 + 순서 완전 일치 | 엄격한 품질 보증 |
| `unordered` | 재료만 같으면 OK | 도구 집합 일치 (순서 무관) | 병렬 호출 가능한 에이전트 |
| `subset` | 최소 재료 포함 확인 | 기대 궤적이 실제의 부분집합 | 에이전트가 추가 탐색 허용 |
| `superset` | 필수 단계 보장 | 실제 궤적이 기대의 상위집합 | 최소 필수 도구 보장 |

> 🎯 **강의 포인트**: `strict` 모드로 시작해서 실패 원인을 분석하고, 그에 맞는 완화 모드를 선택하는 것이 좋아요. 처음부터 `unordered`를 쓰면 순서가 중요한 버그를 놓칠 수 있어요.

> 💡 **실무 팁**: 대부분의 실무 프로젝트에서는 `unordered` + `tool_args_match_mode="ignore"`로 시작해요. 에이전트가 도시 이름을 "서울" vs "Seoul"로 다르게 전달하는 경우가 많기 때문이에요.

In [13]:
# !uv add agentevals

In [14]:
# agentevals에서 궤적 매칭 평가자를 가져와요
from agentevals.trajectory.match import create_trajectory_match_evaluator

In [15]:
# ---------------------------------------------------
# 궤적 매처 생성: 4가지 모드 비교
# ---------------------------------------------------
# 파라미터: trajectory_match_mode (v0.0.9 기준 API)

# strict: 도구 이름 + 인자 + 순서 완전 일치를 요구해요
strict_evaluator = create_trajectory_match_evaluator(trajectory_match_mode="strict")

# unordered: 어떤 도구를 사용했는지만 보고 순서는 무시해요
unordered_evaluator = create_trajectory_match_evaluator(trajectory_match_mode="unordered")

# subset: 기대 궤적의 도구들이 실제 궤적에 포함되어 있으면 통과해요
subset_evaluator = create_trajectory_match_evaluator(trajectory_match_mode="subset")

# superset: 실제 궤적이 기대 궤적을 포함하면 통과해요 (추가 도구 허용)
superset_evaluator = create_trajectory_match_evaluator(trajectory_match_mode="superset")

# 궤적 매처 4종 생성 완료
#   - strict: 완전 일치
#   - unordered: 집합 일치 (순서 무관)
#   - subset: 기대 궤적 ⊆ 실제 궤적
#   - superset: 기대 궤적 ⊇ 실제 궤적

In [16]:
# ---------------------------------------------------
# 궤적 매처 동작 확인
# ---------------------------------------------------
# 직접 호출해서 strict(순서 민감)와 unordered(순서 무관)의 차이를 확인해봐요.
#
# [중요] agentevals는 도구 호출을 "assistant 메시지의 tool_calls 필드"에서만 추출해요.
#        {"role": "tool", "tool": ...} 같은 임의 형식은 도구 호출 0개로 인식되고,
#        그러면 "빈 궤적 vs 빈 궤적"이라 어떤 모드든 항상 통과(False Positive)해버려요.
#        그래서 실제 에이전트 메시지와 동일한 형식(tool_calls를 가진 AIMessage)으로 작성해요.
from langchain.messages import AIMessage

# strict는 "메시지 위치"로 순서를 비교하므로 도구 호출을 하나씩 별도 AIMessage로 표현해요
# 실제 궤적: search_weather → get_population
actual_trajectory = [
    AIMessage(content="", tool_calls=[{"name": "search_weather", "args": {"city": "서울"}, "id": "a1"}]),
    AIMessage(content="", tool_calls=[{"name": "get_population", "args": {"city": "서울"}, "id": "a2"}]),
]

# 기대 궤적: get_population → search_weather (순서 반대)
expected_trajectory_reversed = [
    AIMessage(content="", tool_calls=[{"name": "get_population", "args": {"city": "서울"}, "id": "r1"}]),
    AIMessage(content="", tool_calls=[{"name": "search_weather", "args": {"city": "서울"}, "id": "r2"}]),
]

print("== 순서가 다른 궤적 비교 ==")
print("실제: search_weather → get_population / 기대: get_population → search_weather\n")

# strict: 순서가 다르므로 실패(False) 예상
r_strict = strict_evaluator(
    outputs={"messages": actual_trajectory},
    reference_outputs={"messages": expected_trajectory_reversed},
)
print(f"strict    결과: {r_strict['score']}  (순서가 달라 실패 예상)")

# unordered: 도구 집합이 같으므로 통과(True) 예상
r_unordered = unordered_evaluator(
    outputs={"messages": actual_trajectory},
    reference_outputs={"messages": expected_trajectory_reversed},
)
print(f"unordered 결과: {r_unordered['score']}  (집합이 같아 통과 예상)")

== 순서가 다른 궤적 비교 ==
실제: search_weather → get_population / 기대: get_population → search_weather

strict    결과: False  (순서가 달라 실패 예상)
unordered 결과: True  (집합이 같아 통과 예상)


In [17]:
# ---------------------------------------------------
# tool_args_match_mode: 도구 인자 비교 방식
# ---------------------------------------------------
# 기본값 "exact": 도구 이름 + 인자까지 완전히 일치해야 통과
# "ignore": 도구 이름만 비교하고 인자는 무시 (도시 이름 표기 차이에 유용)

# 인자를 무시하는 궤적 매처 생성
ignore_args_evaluator = create_trajectory_match_evaluator(
    trajectory_match_mode="strict",
    tool_args_match_mode="ignore",  # 도구 인자 무시
)

# 테스트: 도구는 같지만 인자(city)가 다른 궤적
# (cell-22와 동일하게 tool_calls를 가진 AIMessage 형식이어야 실제로 비교돼요)
actual_with_diff_arg = [
    AIMessage(content="", tool_calls=[{"name": "search_weather", "args": {"city": "Seoul"}, "id": "a1"}]),  # 영문
]
expected_korean_arg = [
    AIMessage(content="", tool_calls=[{"name": "search_weather", "args": {"city": "서울"}, "id": "r1"}]),    # 한글
]

print("== 인자 비교 모드 테스트 ==")
print("실제: city='Seoul' / 기대: city='서울'\n")

r_exact = strict_evaluator(
    outputs={"messages": actual_with_diff_arg},
    reference_outputs={"messages": expected_korean_arg},
)
r_ignore = ignore_args_evaluator(
    outputs={"messages": actual_with_diff_arg},
    reference_outputs={"messages": expected_korean_arg},
)

print(f"exact  모드 (기본):     {r_exact['score']}  (인자가 달라 실패 예상)")
print(f"ignore 모드 (인자 무시): {r_ignore['score']}  (이름만 같으면 통과)")

== 인자 비교 모드 테스트 ==
실제: city='Seoul' / 기대: city='서울'

exact  모드 (기본):     False  (인자가 달라 실패 예상)
ignore 모드 (인자 무시): True  (이름만 같으면 통과)


## 5. 궤적 LLM-as-Judge 평가

규칙 기반 매처 외에, **LLM이 궤적의 품질을 직접 판단**하도록 할 수도 있어요.

규칙으로 잡기 어려운 "에이전트가 논리적으로 올바른 도구를 선택했는가?"를 평가할 때 유용해요.

> 💡 **실무 팁**: LLM-as-Judge는 비용이 발생해요. 전체 데이터셋에 적용하기 전에 소량 샘플로 먼저 테스트해서 프롬프트를 조정하세요.

In [18]:
# agentevals에서 LLM 기반 궤적 평가자를 가져와요
from agentevals.trajectory.llm import (
    create_trajectory_llm_as_judge,
    TRAJECTORY_ACCURACY_PROMPT,  # prompts 모듈이 아닌 llm 모듈에 있어요
)

In [19]:
# ---------------------------------------------------
# 궤적 LLM-as-Judge 생성
# ---------------------------------------------------
# TRAJECTORY_ACCURACY_PROMPT: 궤적이 목표 달성에 적절한지 평가해요

trajectory_judge = create_trajectory_llm_as_judge(
    prompt=TRAJECTORY_ACCURACY_PROMPT,
    model="openai:gpt-4o-mini",
    feedback_key="trajectory_accuracy",
)

# 궤적 LLM-as-Judge 생성 완료
print(f"사용 프롬프트: TRAJECTORY_ACCURACY_PROMPT")
print(f"사용 모델: gpt-4o-mini")

사용 프롬프트: TRAJECTORY_ACCURACY_PROMPT
사용 모델: gpt-4o-mini


In [20]:
# ---------------------------------------------------
# 궤적 LLM-as-Judge 테스트
# ---------------------------------------------------
# 실제 궤적(에이전트 메시지 히스토리)을 전달해서 평가해봐요

# 도구 호출이 포함된 메시지 히스토리 시뮬레이션
from langchain.messages import HumanMessage, AIMessage, ToolMessage

# 에이전트가 실제로 생성했을 법한 메시지 시퀀스
sample_trajectory = [
    HumanMessage(content="서울 날씨랑 인구를 알려줘"),
    AIMessage(
        content="",
        tool_calls=[
            {"name": "search_weather", "args": {"city": "서울"}, "id": "tc1"},
        ]
    ),
    ToolMessage(content="맑음, 기온 22도", tool_call_id="tc1"),
    AIMessage(
        content="",
        tool_calls=[
            {"name": "get_population", "args": {"city": "서울"}, "id": "tc2"},
        ]
    ),
    ToolMessage(content="약 950만 명", tool_call_id="tc2"),
    AIMessage(content="서울은 맑고 22도이며, 인구는 약 950만 명이에요."),
]

# 기대 궤적 (데이터셋의 reference output)
reference_trajectory = [
    {"role": "tool", "tool": "search_weather", "tool_input": {"city": "서울"}},
    {"role": "tool", "tool": "get_population", "tool_input": {"city": "서울"}},
]

# LLM으로 궤적 정확도 평가
# agentevals v0.0.9: reference_outputs는 "messages" 키를 사용해요
trajectory_score = trajectory_judge(
    inputs={"messages": [{"role": "user", "content": "서울 날씨랑 인구를 알려줘"}]},
    outputs={"messages": sample_trajectory},
    reference_outputs={"messages": reference_trajectory},
)
print(f"궤적 정확도 점수: {trajectory_score}")

궤적 정확도 점수: {'key': 'trajectory_accuracy', 'score': True, 'comment': "The trajectory effectively addresses the user's request for information about the weather and population of Seoul. Each step logically builds upon the previous one, starting with the user asking for information, followed by two tool calls to fetch the respective data (weather and population), and finally, delivering a cohesive response that summarizes both pieces of information. This shows clear progression and logical coherence. The process is also relatively efficient, utilizing two distinct tool calls to obtain the necessary information rather than attempting to consolidate these into a single request. Thus, the score should be: true.", 'metadata': None}


## 6. 전체 평가 파이프라인 실행

이제 모든 평가자를 조합해서 LangSmith로 **전체 평가 파이프라인**을 실행해요.

`client.evaluate()`는:
1. 데이터셋의 각 예제를 가져와요
2. `target` 함수(에이전트)에 입력을 전달해서 출력을 얻어요
3. 각 `evaluators`를 실행해서 점수를 매겨요
4. 결과를 LangSmith에 실험(experiment)으로 기록해요

```mermaid
flowchart LR
    DS[(데이터셋)] --> |"예제 N개"| CE["client.evaluate()"]
    CE --> |"각 예제마다"| TG["target(inputs)"]
    TG --> |outputs| EV1["평가자 1<br/>correctness"]
    TG --> |outputs| EV2["평가자 2<br/>trajectory"]
    TG --> |outputs| EV3["평가자 3<br/>custom"]
    EV1 --> LS[(LangSmith<br/>실험 결과)]
    EV2 --> LS
    EV3 --> LS

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class DS input
    class CE,TG process
    class EV1,EV2,EV3 output
    class LS storage
```

> 🔑 **핵심 개념**: `target` 함수는 데이터셋 `inputs`를 받아 평가자가 사용할 `outputs`를 반환해야 해요. 에이전트 출력 형식을 이 함수 안에서 정규화하는 게 좋아요.

> ⚠️ **자주 하는 실수**: `client.evaluate()`의 첫 번째 인자는 **위치 인자(positional)**예요. `target=run_agent`가 아니라 `run_agent`로 직접 전달해야 해요. `langsmith` 0.7.x부터 변경된 부분이에요.

### 이번 실험에서 점수가 "갈리도록" 설계한 이유

평가가 의미 있으려면 모든 점수가 1.0이면 안 돼요. 데이터셋(섹션 2)에 일부러 두 가지 함정 케이스를 넣었어요. 이 둘이 **correctness와 trajectory가 서로 다른 문제를 잡는다**는 걸 보여줘요.

| 예제 | 에이전트 행동 | correctness | strict / unordered | superset | 교훈 |
|------|---------------|:-----------:|:------------------:|:--------:|------|
| 정상 4종 | 올바른 도구 → 올바른 답 | 1 | 1 | 1 | 기준점 |
| **대전 날씨** | 올바른 도구 호출, 그러나 데이터 없음 → 답 못 줌 | **0** | 1 | 1 | 궤적은 맞아도 정답이 틀릴 수 있다 (길은 맞는데 목적지 도착 실패) |
| **서울 전체** | 정답은 맞지만 기대보다 도구를 더 호출 | 1 | **0** | **1** | 정답이 맞아도 궤적이 어긋날 수 있다 (불필요한 우회) |

> 🔑 **핵심**: 최종 응답만 보면 '대전'을 놓치고, 궤적만 보면 '서울 전체'를 과하게 깎아요. 그래서 두 평가를 **함께** 써야 해요. `superset`은 "필수 도구만 다 호출했으면 추가는 허용"이라 '서울 전체'를 통과시켜요 — 평가 정책을 얼마나 엄격하게 둘지는 프로젝트가 정해요.

> ⚙️ **재현성**: 에이전트(gpt-4o-mini)와 심판(gpt-4o) 모두 `temperature=0`이라 여러 번 실행해도 같은 점수가 나와요. 무작위성을 제거하지 않으면 위 표가 실행마다 흔들려서 "설계된 다양성"과 "노이즈"를 구분할 수 없어요.


In [21]:
# ---------------------------------------------------
# 평가 대상 함수 (target) 정의
# ---------------------------------------------------
# target 함수: 데이터셋 inputs → 에이전트 실행 → outputs 반환
# LangSmith evaluate()가 각 예제마다 이 함수를 호출해요

def run_agent(inputs: dict) -> dict:
    """데이터셋 입력으로 에이전트를 실행하고 결과를 반환해요.
    
    Args:
        inputs: 데이터셋 예제의 inputs 필드
                예: {"messages": [{"role": "user", "content": "..."}]}
    
    Returns:
        에이전트 실행 결과 (messages 포함)
    """
    # 딕셔너리 형태의 메시지를 LangChain Message 객체로 변환
    from langchain.messages import HumanMessage as HM
    
    raw_messages = inputs.get("messages", [])
    messages = []
    for msg in raw_messages:
        if isinstance(msg, dict) and msg.get("role") == "user":
            messages.append(HM(content=msg["content"]))
        else:
            messages.append(msg)  # 이미 Message 객체인 경우
    
    # 에이전트 실행
    result = agent.invoke({"messages": messages})
    
    # 평가자가 사용할 수 있는 형식으로 반환
    return result


# 함수 동작 확인
sample_input = {"messages": [{"role": "user", "content": "서울 날씨 알려줘"}]}
sample_output = run_agent(sample_input)
# target 함수 테스트:
print(f"  입력: {sample_input['messages'][0]['content']}")
print(f"  출력 메시지 수: {len(sample_output['messages'])}")
print(f"  최종 응답: {sample_output['messages'][-1].content}")

  입력: 서울 날씨 알려줘
  출력 메시지 수: 4
  최종 응답: 현재 서울의 날씨는 맑고, 기온은 22도입니다.


In [22]:
# ---------------------------------------------------
# 평가자 함수: 궤적 매칭 래퍼
# ---------------------------------------------------
# LangSmith evaluate()와 호환되는 형식으로 래핑해요.
#
# [핵심] 데이터셋 trajectory는 {"tool": ..., "tool_input": ...} 형식이지만,
#        agentevals는 도구 호출을 "assistant 메시지의 tool_calls 필드"에서만 추출해요.
#        따라서 비교 전에 양쪽을 "도구 호출 1개 = AIMessage 1개" 형식으로 정규화해요.
#        이렇게 하면 에이전트가 도구를 병렬로 부르든 순차로 부르든 결과가 같아져요.
from langchain.messages import AIMessage


def _extract_calls(messages: list) -> list:
    """메시지 히스토리에서 도구 호출(tool_calls)을 호출 순서대로 모두 추출해요."""
    calls = []
    for msg in messages:
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls is None and isinstance(msg, dict):
            tool_calls = msg.get("tool_calls")
        for tc in (tool_calls or []):
            calls.append(tc)
    return calls


def _expected_calls(trajectory: list) -> list:
    """데이터셋의 기대 궤적([{tool, tool_input}, ...])을 tool_calls 형식으로 바꿔요."""
    return [
        {"name": step["tool"], "args": step.get("tool_input", {}), "id": f"ref_{i}"}
        for i, step in enumerate(trajectory)
    ]


def _calls_to_messages(calls: list) -> list:
    """도구 호출 목록을 '호출당 AIMessage 1개' 형식으로 변환해요.
    strict 모드가 메시지 위치로 순서를 비교할 수 있게 하기 위함이에요."""
    return [AIMessage(content="", tool_calls=[c]) for c in calls]


def _run_match(evaluator, outputs: dict, reference_outputs: dict) -> dict:
    """실제 출력과 기대 궤적을 정규화한 뒤 주어진 매처로 채점해요."""
    actual = _calls_to_messages(_extract_calls(outputs.get("messages", [])))
    expected = _calls_to_messages(_expected_calls(reference_outputs.get("trajectory", [])))
    return evaluator(
        outputs={"messages": actual},
        reference_outputs={"messages": expected},
    )


def evaluate_trajectory_strict(outputs: dict, reference_outputs: dict) -> dict:
    """strict: 도구 이름 + 인자 + 순서까지 완전히 일치해야 통과."""
    return _run_match(strict_evaluator, outputs, reference_outputs)


def evaluate_trajectory_unordered(outputs: dict, reference_outputs: dict) -> dict:
    """unordered: 순서는 무시하되 도구 호출 '집합'이 정확히 같아야 통과."""
    return _run_match(unordered_evaluator, outputs, reference_outputs)


def evaluate_trajectory_superset(outputs: dict, reference_outputs: dict) -> dict:
    """superset: 실제 궤적이 기대 궤적을 '포함'하면 통과 (추가 도구 허용).
    기대한 필수 도구를 모두 호출했는지 보장하고 싶을 때 써요."""
    return _run_match(superset_evaluator, outputs, reference_outputs)


# 평가자 함수 래퍼 정의 완료 (strict / unordered / superset)

In [23]:
# ---------------------------------------------------
# 전체 평가 실행
# ---------------------------------------------------
# client.evaluate(): 데이터셋의 모든 예제에 대해 평가를 실행해요
#
# 파라미터:
#   첫 번째 인자(positional): 에이전트 실행 함수 (inputs → outputs)
#   data: 평가할 데이터셋 이름 또는 ID
#   evaluators: 평가자 함수 목록 (각각 outputs, reference_outputs 받음)
#   experiment_prefix: LangSmith 실험 이름 접두사
#
# 주의: langsmith 0.7.x에서 target은 위치 인자(positional-only)예요

# 평가 파이프라인 실행 시작...
print(f"데이터셋: {DATASET_NAME}")
print("평가자: correctness, trajectory_strict, trajectory_unordered, trajectory_superset")
print()

results = client.evaluate(
    run_agent,  # target은 위치 인자로 전달해요
    data=DATASET_NAME,
    evaluators=[
        evaluate_correctness,          # 최종 응답 정확도 (LLM-as-Judge)
        evaluate_trajectory_strict,    # 궤적: 순서까지 완전 일치
        evaluate_trajectory_unordered, # 궤적: 순서 무관, 집합 일치
        evaluate_trajectory_superset,  # 궤적: 필수 도구 포함 (추가 도구 허용)
    ],
    experiment_prefix="도시정보-에이전트-v1",
    max_concurrency=2,  # 동시 실행 수 제한 (API 레이트 리밋 방지)
)

# 평가 완료!
print(f"실험 URL: {results.url}")

데이터셋: 도시정보-에이전트-평가-v1
평가자: correctness, trajectory_strict, trajectory_unordered, trajectory_superset



/Users/mhso/working/lecture/llm/langchain-llm/03_langgraph-agents/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: '도시정보-에이전트-v1-d0267b63' at:
https://smith.langchain.com/o/7f9bfcb6-cd10-4707-bfdd-8aeff8fa7bb6/datasets/845b0547-f6ff-449c-870f-9c2a003c8148/compare?selectedSessions=a47226c0-b84f-48cf-a623-4b5a1e935dd5




6it [00:09,  1.66s/it]

실험 URL: https://smith.langchain.com/o/7f9bfcb6-cd10-4707-bfdd-8aeff8fa7bb6/datasets/845b0547-f6ff-449c-870f-9c2a003c8148/compare?selectedSessions=a47226c0-b84f-48cf-a623-4b5a1e935dd5


In [24]:
# ---------------------------------------------------
# 평가 결과 요약 출력 (예제별 분포 + 전체 평균)
# ---------------------------------------------------
# 점수가 "다양하게" 나오는지 예제별로 펼쳐서 확인해요.

import pandas as pd

results_df = results.to_pandas()
print(f"총 예제 수: {len(results_df)}\n")

# 점수 컬럼 (feedback.으로 시작) 추출
score_columns = [col for col in results_df.columns if col.startswith("feedback.")]


def _question_of(row) -> str:
    """행에서 사용자 질문 텍스트를 최대한 안전하게 꺼내요."""
    for key in ("inputs.messages", "inputs.inputs.messages"):
        if key in row and isinstance(row[key], (list, tuple)) and row[key]:
            first = row[key][0]
            if isinstance(first, dict):
                return first.get("content", "")
    return "(질문 추출 실패)"


if score_columns:
    # 1) 예제별 점수 표
    print("=== 예제별 점수 ===")
    short = lambda name: name.replace("feedback.", "").replace("trajectory_", "").replace("_match", "")
    header = "  " + "질문".ljust(26) + "".join(short(c).rjust(12) for c in score_columns)
    print(header)
    print("  " + "-" * (len(header) - 2))
    for _, row in results_df.iterrows():
        q = _question_of(row)[:24]
        cells = ""
        for c in score_columns:
            v = row[c]
            cells += ("-" if pd.isna(v) else f"{float(v):.0f}").rjust(12)
        print("  " + q.ljust(26) + cells)

    # 2) 전체 평균
    print("\n=== 전체 평균 ===")
    for c in score_columns:
        print(f"  {short(c)}: {results_df[c].mean():.2f}")

    print("\n해석: '대전'은 궤적O·정답X로 correctness만 0, "
          "'서울 전체'는 정답O·궤적X로 strict/unordered만 0이고 superset은 1이에요.")
else:
    print(f"점수 데이터가 아직 처리 중이에요. LangSmith에서 확인하세요: {results.url}")

총 예제 수: 6

=== 예제별 점수 ===
  질문                         correctness      strict   unordered    superset
  --------------------------------------------------------------------------
  제주의 날씨와 인구를 모두 알려줘                   1           1           1           1
  서울의 대표 랜드마크는 뭐가 있어?                  1           1           1           1
  부산의 인구가 얼마나 돼?                       1           1           1           1
  서울의 날씨, 인구, 랜드마크를 모두 알려줘             1           0           0           1
  대전의 날씨를 알려줘                          0           1           1           1
  서울의 날씨를 알려줘                          1           1           1           1

=== 전체 평균 ===
  correctness: 0.83
  strict: 0.83
  unordered: 0.83
  superset: 1.00

해석: '대전'은 궤적O·정답X로 correctness만 0, '서울 전체'는 정답O·궤적X로 strict/unordered만 0이고 superset은 1이에요.


## 7. 커스텀 평가자 작성

라이브러리가 제공하는 평가자 외에, **프로젝트 특화 규칙**을 직접 구현할 수 있어요.

평가자 함수의 시그니처만 맞추면 어떤 로직이든 사용할 수 있어요.

> 💡 **실무 팁**: 커스텀 평가자는 도메인 지식이 필요한 규칙을 구현할 때 강력해요. 예를 들어 "의료 챗봇은 반드시 면책 조항을 포함해야 한다"는 규칙은 LLM 심판보다 코드로 구현하는 게 더 정확하고 비용이 낮아요.

In [25]:
# ---------------------------------------------------
# 커스텀 평가자 1: 도구 호출 횟수 검증
# ---------------------------------------------------
# 에이전트가 적절한 횟수의 도구를 호출했는지 검증해요
# 너무 많이 호출하면 비효율, 너무 적으면 정보 부족

def evaluate_tool_call_count(outputs: dict, reference_outputs: dict) -> dict:
    """에이전트가 예상 횟수만큼 도구를 호출했는지 평가해요.
    
    Args:
        outputs: 에이전트 실행 출력 (messages 포함)
        reference_outputs: 기대 출력 (trajectory 포함)
    
    Returns:
        {'tool_call_count_match': 1.0} 또는 {'tool_call_count_match': 0.0}
    """
    # 실제 도구 호출 횟수 계산 (ToolMessage 수)
    actual_messages = outputs.get("messages", [])
    from langchain.messages import ToolMessage as TM
    actual_tool_calls = sum(
        1 for msg in actual_messages
        if isinstance(msg, TM)
    )
    
    # 기대 도구 호출 횟수
    expected_trajectory = reference_outputs.get("trajectory", [])
    expected_tool_calls = len(expected_trajectory)
    
    # 일치 여부 반환
    match = 1.0 if actual_tool_calls == expected_tool_calls else 0.0
    
    return {
        "tool_call_count_match": match,
        # 디버깅 정보도 함께 반환할 수 있어요
        "comment": f"실제: {actual_tool_calls}회, 기대: {expected_tool_calls}회"
    }


# 커스텀 평가자: evaluate_tool_call_count 정의 완료

In [26]:
# ---------------------------------------------------
# 커스텀 평가자 2: 응답 언어 검증
# ---------------------------------------------------
# 한국어 질문에는 한국어로 답했는지 간단히 검증해요

def evaluate_response_language(outputs: dict, reference_outputs: dict) -> dict:
    """에이전트가 한국어로 응답했는지 평가해요.
    
    Args:
        outputs: 에이전트 실행 출력
        reference_outputs: 기대 출력 (사용하지 않음)
    
    Returns:
        {'korean_response': 1.0} 한국어 응답 여부
    """
    # 최종 응답 텍스트 추출
    messages = outputs.get("messages", [])
    if not messages:
        return {"korean_response": 0.0}
    
    last_message = messages[-1]
    if hasattr(last_message, "content"):
        response_text = last_message.content
    else:
        response_text = str(last_message)
    
    # 한글 문자 포함 여부로 판단 (유니코드 범위: 가-힣)
    has_korean = any('가' <= char <= '힣' for char in response_text)
    
    return {"korean_response": 1.0 if has_korean else 0.0}


# 커스텀 평가자: evaluate_response_language 정의 완료

In [27]:
# ============================================================
# TODO: 커스텀 평가자로 전체 평가 실행
# 힌트: client.evaluate()의 evaluators 파라미터에
#       evaluate_tool_call_count와 evaluate_response_language를 추가해요
# 예상 결과: LangSmith에 tool_call_count_match, korean_response 점수가 추가돼요
# ============================================================

# 커스텀 평가자 포함 평가 실행...

custom_results = client.evaluate(
    run_agent,  # target은 위치 인자로 전달해요
    data=DATASET_NAME,
    evaluators=[
        evaluate_correctness,
        evaluate_tool_call_count,     # TODO: 이 평가자의 역할은?
        evaluate_response_language,   # TODO: 이 평가자의 역할은?
    ],
    experiment_prefix="도시정보-커스텀-평가자",
    max_concurrency=2,
)

print(f"\n커스텀 평가 완료!")
print(f"결과 확인: {custom_results.url}")

View the evaluation results for experiment: '도시정보-커스텀-평가자-46d232db' at:
https://smith.langchain.com/o/7f9bfcb6-cd10-4707-bfdd-8aeff8fa7bb6/datasets/845b0547-f6ff-449c-870f-9c2a003c8148/compare?selectedSessions=6960ae76-b062-40ac-b1e8-d1487d8b3a4e




0it [00:00, ?it/s]Error running evaluator <DynamicRunEvaluator evaluate_tool_call_count> on run 019eed7a-73ea-7203-9e8f-6888c5288e77: ValueError("Expected an EvaluationResult object, or dict with a metric 'key' and optional 'score'; got {'tool_call_count_match': 1.0, 'comment': '실제: 1회, 기대: 1회', 'key': 'evaluate_tool_call_count'}")
Traceback (most recent call last):
  File "/Users/mhso/working/lecture/llm/langchain-llm/03_langgraph-agents/.venv/lib/python3.14/site-packages/langsmith/evaluation/evaluator.py", line 254, in _coerce_evaluation_result
    return EvaluationResult(**{"source_run_id": source_run_id, **result})
  File "/Users/mhso/working/lecture/llm/langchain-llm/03_langgraph-agents/.venv/lib/python3.14/site-packages/pydantic/main.py", line 263, in __init__
    validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
pydantic_core._pydantic_core.ValidationError: 1 validation error for EvaluationResult
tool_call_count_match
  Extra inputs are not p


커스텀 평가 완료!
결과 확인: https://smith.langchain.com/o/7f9bfcb6-cd10-4707-bfdd-8aeff8fa7bb6/datasets/845b0547-f6ff-449c-870f-9c2a003c8148/compare?selectedSessions=6960ae76-b062-40ac-b1e8-d1487d8b3a4e


## 8. 평가 결과 분석 및 활용

평가가 끝나면 결과를 분석해서 에이전트를 개선하는 방향을 찾아요.

```mermaid
flowchart LR
    EV[평가 실행] --> |점수 기록| LS[LangSmith 실험]
    LS --> |결과 분석| AN{분석}
    AN --> |낮은 궤적 점수| FT[도구 선택 개선]
    AN --> |낮은 정확도 점수| PR[프롬프트 개선]
    AN --> |언어 오류| SY[시스템 프롬프트 수정]
    FT --> |반영| AG2[개선된 에이전트]
    PR --> AG2
    SY --> AG2
    AG2 --> |재평가| EV

    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class EV,FT,PR,SY process
    class AN output
    class LS storage
    class AG2 storage
```

> 🎯 **강의 포인트**: 평가는 한 번으로 끝나는 게 아니에요. 에이전트를 개선할 때마다 `experiment_prefix`를 바꿔서 새 실험을 만들고, LangSmith에서 실험들을 **비교**하는 워크플로우가 핵심이에요.

> ⚠️ **자주 하는 실수**: 평가 점수가 낮다고 바로 모델을 바꾸는 게 아니에요. 먼저 **어떤 예제에서 실패했는지** 확인하고 패턴을 찾아야 해요.

In [28]:
# ---------------------------------------------------
# 실패한 예제 분석
# ---------------------------------------------------
# 점수가 낮은 예제를 찾아서 에이전트 개선 방향을 파악해요

# === 결과 분석 ===
print()

# custom_results에서 데이터 가져오기
try:
    df = custom_results.to_pandas()
    
    # 점수 컬럼 목록
    feedback_cols = [col for col in df.columns if col.startswith("feedback.")]
    
    if feedback_cols:
        # 전체 평균 점수
        # 전체 평균 점수:
        for col in feedback_cols:
            print(f"  {col.replace('feedback.', '')}: {df[col].mean():.2f}")
        print()
        
        # 실패한 예제 (점수 0인 케이스) 출력
        # 실패 케이스 분석:
        for col in feedback_cols:
            failed = df[df[col] == 0.0]
            if not failed.empty:
                print(f"  [{col.replace('feedback.', '')}] 실패 예제 수: {len(failed)}")
    else:
        # 점수가 아직 처리 중이에요.
        print(f"LangSmith 대시보드에서 확인하세요: {custom_results.url}")

except Exception as e:
    print(f"결과 분석 중 오류: {e}")
    # LangSmith 대시보드에서 직접 확인해주세요.


  correctness: 0.83
  wrapper: nan

  [correctness] 실패 예제 수: 1


### LangSmith 실험 비교 방법

1. LangSmith 대시보드에서 **Datasets & Testing** 탭으로 이동해요
2. 위에서 생성한 데이터셋을 클릭해요
3. 여러 실험을 체크박스로 선택해요
4. **Compare Experiments** 클릭해요

비교 화면에서는 각 평가자별 점수 차이, 응답 시간, 실패율 등을 한눈에 확인할 수 있어요.

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **3가지 평가 전략**: 최종 응답 평가(요리 맛), 궤적 평가(조리 과정), 단일 스텝 평가(개별 단계) -- 목적에 따라 적절한 전략을 선택해요
- **agentevals 궤적 매처**: `strict`, `unordered`, `subset`, `superset` 4가지 모드로 도구 호출 시퀀스를 검증해요. `tool_args_match_mode`로 인자 비교 방식도 제어할 수 있어요
- **openevals LLM-as-Judge**: `CORRECTNESS_PROMPT`로 텍스트의 의미론적 정확도를 자동 평가해요. "22도"와 "섭씨 22도"처럼 표현이 달라도 의미가 같으면 통과해요
- **LangSmith 평가 파이프라인**: `client.create_dataset()` → `client.create_examples()` → `client.evaluate()`의 3단계 워크플로우
- **커스텀 평가자**: `(outputs, reference_outputs) → dict` 시그니처로 도메인 특화 평가 규칙을 직접 구현할 수 있어요
- **반복적 평가**: `experiment_prefix`로 실험을 구분하고 LangSmith에서 비교해서 에이전트를 점진적으로 개선해요

### 테스트와 평가의 관계 (Part 12 전체 요약)

```mermaid
flowchart LR
    UT["단위 테스트<br/>01-Agent-Testing"] --> IT["통합 테스트<br/>01-Agent-Testing"]
    IT --> EV["평가<br/>02-Agent-Evaluation"]
    EV --> DP["배포<br/>Part 13"]

    UT -.->|"버그 발견"| FIX1["코드 수정"]
    IT -.->|"통합 문제"| FIX2["설정 수정"]
    EV -.->|"품질 미달"| FIX3["프롬프트 개선"]

    classDef test fill:#d4edda,stroke:#28a745,color:#155724
    classDef eval fill:#cce5ff,stroke:#007bff,color:#004085
    classDef deploy fill:#fff3cd,stroke:#ffc107,color:#856404

    class UT,IT test
    class EV eval
    class DP deploy
```

## 다음 노트북 예고

다음 `03-Generator-Evaluator-Verification.ipynb`에서는 **자체 평가 함정과 검증 패턴**을 배워요. 동일한 모델이 생성과 평가를 동시에 하면 확증 편향이 생기는 문제를, `ast.parse()` 같은 **computational verifier**와 Pydantic 기반 **inferential verifier**로 분리하는 방법을 다뤄요. Anthropic이 명시적으로 경고한 "자기 평가 편향"을 제거하는 실전 패턴이에요.